# ML-03 — Frame Your Lane as an ML Task

**Lane:** human-reviewed content-refresh prioritization. This notebook uses only the repository's anonymized starter data; it contains no client names, URLs, or raw queries.

## 1. My lane as an ML task

This is a **ranking / scoring** task. A content lead needs a short queue of pages to inspect first when there are more candidates than editorial hours. The system ranks candidates; a person decides whether any page should be revised.

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path('data/raw/content_refresh_anonymized.csv')
if not DATA.exists():
    DATA = Path.cwd().parents[1] / 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA)
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = df['trend_direction'].eq('down').astype('int8')

print(f'rows={len(df):,}; pseudonymized client groups={df.client_id.nunique()}')

rows=30,000; pseudonymized client groups=32


## 2. Target or proxy

The starter-data proxy is `is_declining_label`, derived from observed `trend_direction == \"down\"`. It is useful for a teaching-slice ranking comparison, but it is contemporaneous rather than a future causal outcome. Therefore this work claims only observed, directional decision support; `trend_direction` and `trend_pct` are excluded as features.

In [2]:
from pathlib import Path
import pandas as pd

DATA = Path('data/raw/content_refresh_anonymized.csv')
if not DATA.exists():
    DATA = Path.cwd().parents[1] / 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA)
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = df['trend_direction'].eq('down').astype('int8')

print('decline-proxy base rate=', round(df.is_declining_label.mean(), 3))
print('label source: observed trend direction; not a future recovery outcome')

decline-proxy base rate= 0.542
label source: observed trend direction; not a future recovery outcome


## 3. Success metric

The primary metric is **Precision@50**: of the 50 pages placed at the top of the review queue, how many match the decline proxy? It matches a bounded weekly review capacity. The capstone's grouped-client comparison measured 0.84 for the cleaned model versus 0.32 for the transparent rule baseline on the same held-out setup.

In [3]:
baseline_p50, model_p50 = 0.32, 0.84
print(f'Precision@50 baseline={baseline_p50:.2f}; cleaned model={model_p50:.2f}')

Precision@50 baseline=0.32; cleaned model=0.84


## 4. The unit of analysis

One row is one anonymized content item at a snapshot. `client_id` is used only to hold out whole client groups during validation; neither it nor `content_id` is a model feature.

In [4]:
from pathlib import Path
import pandas as pd

DATA = Path('data/raw/content_refresh_anonymized.csv')
if not DATA.exists():
    DATA = Path.cwd().parents[1] / 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA)
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = df['trend_direction'].eq('down').astype('int8')

print('one row = one anonymized content item')
print('columns=', len(df.columns), '| content IDs unique=', df.content_id.nunique() == len(df))

one row = one anonymized content item
columns= 45 | content IDs unique= True


## 5. Why ML beats a fixed rule here

A simple rule is necessary as a transparent baseline, but it cannot combine non-linear interactions among visibility, position, staleness, content age, engagement, and metadata. ML earns its place only if it improves the top-of-queue ranking on unseen client groups. The output remains a review agenda—not autonomous rewriting, publishing, deletion, or a guarantee of recovery.

## Self-check

- [x] Decision, actor, cost, target boundary, and metric are stated.
- [x] Code uses the anonymized starter slice and runs top to bottom.
- [x] No private data is displayed.
- [x] Claims are decision-support only.